# P10.6-AI - RSNA dataset preflight

Preflight **train-only** para RSNA LumbarDISC. Descarga y extrae el conjunto de entrenamiento directamente en el disco local de Colab, ejecuta inventario, distribuciones y validaciones sanitizadas, y guarda únicamente los reportes en Google Drive.

No entrena, no accede al test oficial y no genera diagnóstico clínico.


## Guardias operativas

- `humanReviewRequired=true` y `notClinicalDiagnosis=true`.
- El conjunto oficial de test no se extrae ni se inspecciona.
- `disc_bulge`, `disc_protrusion`, `disc_extrusion` y `disc_sequestration` quedan excluidas de entrenamiento con RSNA.
- El token de Kaggle se solicita de forma oculta y se elimina del entorno al terminar la descarga.
- Los DICOM y CSV originales no se versionan.
- Los reportes se escriben fuera de Git bajo `PFI_P10_6_OUTPUT_ROOT`.


In [20]:
from pathlib import Path
import shutil

shutil.rmtree(
    Path("/content/RSNA_LUMBAR_DISC"),
    ignore_errors=True,
)

shutil.rmtree(
    Path("/content/rsna_kaggle_local"),
    ignore_errors=True,
)

print("Copia local parcial eliminada.")

Copia local parcial eliminada.


In [21]:
# 1) Dependencias
from __future__ import annotations
import importlib.util, subprocess, sys

REQUIRED_MODULES = {
    "numpy": "numpy", "pandas": "pandas", "pydicom": "pydicom",
    "SimpleITK": "SimpleITK", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "torch": "torch", "torchvision": "torchvision",
    "timm": "timm", "monai": "monai", "yaml": "pyyaml", "kaggle": "kaggle",
}
missing = [pkg for mod, pkg in REQUIRED_MODULES.items() if importlib.util.find_spec(mod) is None]
if missing:
    try:
        import google.colab  # type: ignore  # noqa: F401
    except Exception as exc:
        raise RuntimeError(f"Dependencias faltantes fuera de Colab: {missing}") from exc
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [22]:
# 2) Imports y entorno
import getpass, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path, PurePosixPath
import pandas as pd
import pydicom
import SimpleITK as sitk
import torch
import yaml
from sklearn.model_selection import StratifiedGroupKFold  # noqa: F401
print({
    "python": sys.version.split()[0], "pytorch": torch.__version__,
    "cudaAvailable": torch.cuda.is_available(), "cudaVersion": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "pydicom": pydicom.__version__, "SimpleITK": sitk.Version_VersionString(),
})


{'python': '3.12.13', 'pytorch': '2.11.0+cpu', 'cudaAvailable': False, 'cudaVersion': None, 'gpu': None, 'pydicom': '3.0.2', 'SimpleITK': '2.5.6'}


In [23]:
# 3) Montar Google Drive solo para reportes y modelos
from google.colab import drive  # type: ignore
drive.mount("/content/drive", force_remount=False)
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
OUTPUT_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
MODEL_ROOT = PFI_ROOT / "models" / "P10_6_rsna_findings"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
print({"outputRoot": str(OUTPUT_ROOT), "modelRoot": str(MODEL_ROOT)})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'outputRoot': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings', 'modelRoot': '/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings'}


## Descarga local optimizada

Descarga el ZIP oficial directamente a `/content` y extrae únicamente `train.csv`, `train_label_coordinates.csv`, `train_series_descriptions.csv` y `train_images/`. No copia miles de archivos desde Drive y no extrae el test oficial.


In [24]:
# 4) Configuración de descarga local
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"
LOCAL_RSNA_ROOT = Path("/content/RSNA_LUMBAR_DISC")
LOCAL_DOWNLOAD_ROOT = Path("/content/rsna_kaggle_local")
LOCAL_TRAIN_IMAGES = LOCAL_RSNA_ROOT / "train_images"
ALLOWED_CSV_NAMES = {"train.csv", "train_label_coordinates.csv", "train_series_descriptions.csv"}
FORCE_LOCAL_REDOWNLOAD = False

total, used, free = shutil.disk_usage("/content")
print({"runtimeTotalGiB": round(total/1024**3,2), "runtimeUsedGiB": round(used/1024**3,2), "runtimeFreeGiB": round(free/1024**3,2)})
if free < 45*1024**3 and not LOCAL_TRAIN_IMAGES.exists():
    raise RuntimeError("Espacio local insuficiente: se requieren aproximadamente 45 GiB libres.")


{'runtimeTotalGiB': 107.72, 'runtimeUsedGiB': 37.12, 'runtimeFreeGiB': 70.58}


In [25]:
# 5) Descargar y extraer RSNA train-only directamente en /content
def local_train_ready() -> bool:
    csv_ready = all((LOCAL_RSNA_ROOT/n).is_file() and (LOCAL_RSNA_ROOT/n).stat().st_size > 0 for n in ALLOWED_CSV_NAMES)
    first_dicom = next(LOCAL_TRAIN_IMAGES.rglob("*.dcm"), None) if LOCAL_TRAIN_IMAGES.is_dir() else None
    return csv_ready and first_dicom is not None

def allowed_relative_path(member_name: str) -> Path | None:
    normalized = PurePosixPath(member_name)
    if normalized.is_absolute() or ".." in normalized.parts:
        raise RuntimeError(f"Ruta insegura dentro del ZIP: {member_name!r}")
    if normalized.name in ALLOWED_CSV_NAMES:
        return Path(normalized.name)
    if "train_images" in normalized.parts:
        i = normalized.parts.index("train_images")
        return Path(*normalized.parts[i:])
    return None

if local_train_ready() and not FORCE_LOCAL_REDOWNLOAD:
    print("RSNA train-only ya está disponible en /content. Se omite la descarga.")
else:
    if FORCE_LOCAL_REDOWNLOAD:
        shutil.rmtree(LOCAL_RSNA_ROOT, ignore_errors=True)
        shutil.rmtree(LOCAL_DOWNLOAD_ROOT, ignore_errors=True)
    LOCAL_RSNA_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
    kaggle_executable = shutil.which("kaggle") or str(Path(sys.executable).parent / "kaggle")
    if not Path(kaggle_executable).exists():
        raise RuntimeError("No se encontró el ejecutable de Kaggle.")
    kaggle_token = os.getenv("KAGGLE_API_TOKEN", "").strip() or getpass.getpass(
        "Pegá tu KAGGLE_API_TOKEN y presioná Enter (el valor no se mostrará): "
    ).strip()
    if not kaggle_token:
        raise RuntimeError("No se ingresó un token de Kaggle.")
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    try:
        subprocess.check_call([kaggle_executable, "competitions", "files", COMPETITION, "--page-size", "20", "--quiet"])
        subprocess.check_call([kaggle_executable, "competitions", "download", COMPETITION, "--path", str(LOCAL_DOWNLOAD_ROOT), "--force"])
    finally:
        os.environ.pop("KAGGLE_API_TOKEN", None)
        kaggle_token = ""
    archives = sorted(LOCAL_DOWNLOAD_ROOT.glob("*.zip"))
    if not archives:
        raise RuntimeError("No se encontró el ZIP descargado.")
    archive_path = archives[0]
    print({"archive": archive_path.name, "archiveSizeGiB": round(archive_path.stat().st_size/1024**3,2)})
    extracted = 0
    root_resolved = LOCAL_RSNA_ROOT.resolve()
    started = time.time()
    with zipfile.ZipFile(archive_path, "r") as archive:
        for member in archive.infolist():
            if member.is_dir():
                continue
            relative_path = allowed_relative_path(member.filename)
            if relative_path is None:
                continue
            destination = (LOCAL_RSNA_ROOT / relative_path).resolve()
            if destination != root_resolved and root_resolved not in destination.parents:
                raise RuntimeError(f"Intento de extracción fuera del destino: {member.filename!r}")
            if destination.exists() and destination.stat().st_size == member.file_size:
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member, "r") as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target, length=8*1024*1024)
            extracted += 1
            if extracted % 5000 == 0:
                print(f"Archivos extraídos: {extracted}")
    print({"newFilesExtracted": extracted, "extractionMinutes": round((time.time()-started)/60,2)})
    shutil.rmtree(LOCAL_DOWNLOAD_ROOT, ignore_errors=True)
    print("ZIP temporal eliminado.")


Pegá tu KAGGLE_API_TOKEN y presioná Enter (el valor no se mostrará): ··········
{'archive': 'rsna-2024-lumbar-spine-degenerative-classification.zip', 'archiveSizeGiB': 28.22}
Archivos extraídos: 5000
Archivos extraídos: 10000
Archivos extraídos: 15000
Archivos extraídos: 20000
Archivos extraídos: 25000
Archivos extraídos: 30000
Archivos extraídos: 35000
Archivos extraídos: 40000
Archivos extraídos: 45000
Archivos extraídos: 50000
Archivos extraídos: 55000
Archivos extraídos: 60000
Archivos extraídos: 65000
Archivos extraídos: 70000
Archivos extraídos: 75000
Archivos extraídos: 80000
Archivos extraídos: 85000
Archivos extraídos: 90000
Archivos extraídos: 95000
Archivos extraídos: 100000
Archivos extraídos: 105000
Archivos extraídos: 110000
Archivos extraídos: 115000
Archivos extraídos: 120000
Archivos extraídos: 125000
Archivos extraídos: 130000
Archivos extraídos: 135000
Archivos extraídos: 140000
Archivos extraídos: 145000
{'newFilesExtracted': 147221, 'extractionMinutes': 8.39}
ZIP t

In [26]:
# 6) Validación local train-only
required = [LOCAL_RSNA_ROOT/n for n in ALLOWED_CSV_NAMES] + [LOCAL_TRAIN_IMAGES]
missing = [str(p) for p in required if not p.exists()]
first_dicom = next(LOCAL_TRAIN_IMAGES.rglob("*.dcm"), None) if LOCAL_TRAIN_IMAGES.is_dir() else None
if first_dicom is None:
    missing.append(f"{LOCAL_TRAIN_IMAGES}/**/*.dcm")
for forbidden_name in ("test_images", "test_series_descriptions.csv", "sample_submission.csv"):
    forbidden_path = LOCAL_RSNA_ROOT / forbidden_name
    if forbidden_path.exists():
        raise RuntimeError(f"Se encontró contenido de test no permitido: {forbidden_path}")
if missing:
    raise RuntimeError("Descarga local incompleta:\n- " + "\n- ".join(missing))
local_dicom_count = sum(1 for _ in LOCAL_TRAIN_IMAGES.rglob("*.dcm"))
print({
    "trainCsv": (LOCAL_RSNA_ROOT/"train.csv").exists(),
    "trainLabelCoordinatesCsv": (LOCAL_RSNA_ROOT/"train_label_coordinates.csv").exists(),
    "trainSeriesDescriptionsCsv": (LOCAL_RSNA_ROOT/"train_series_descriptions.csv").exists(),
    "trainImages": LOCAL_TRAIN_IMAGES.exists(), "localDicomCount": local_dicom_count,
    "officialTestPresent": False, "officialTestAccessed": False, "syntheticMode": False,
})


{'trainCsv': True, 'trainLabelCoordinatesCsv': True, 'trainSeriesDescriptionsCsv': True, 'trainImages': True, 'localDicomCount': 147218, 'officialTestPresent': False, 'officialTestAccessed': False, 'syntheticMode': False}


In [27]:
# 7) Configurar rutas reproducibles
os.environ["PFI_ROOT"] = str(PFI_ROOT)
os.environ["PFI_RSNA_ROOT"] = str(LOCAL_RSNA_ROOT)
os.environ["PFI_RSNA_TRAIN_IMAGES"] = str(LOCAL_TRAIN_IMAGES)
os.environ["PFI_P10_6_OUTPUT_ROOT"] = str(OUTPUT_ROOT)
os.environ["PFI_P10_6_MODEL_ROOT"] = str(MODEL_ROOT)
os.environ["PFI_RSNA_PREFLIGHT_SYNTHETIC"] = "0"
os.environ["PFI_RSNA_HASH_DICOM_OPT_IN"] = "0"
print({
    "PFI_RSNA_ROOT": os.environ["PFI_RSNA_ROOT"],
    "PFI_RSNA_TRAIN_IMAGES": os.environ["PFI_RSNA_TRAIN_IMAGES"],
    "PFI_P10_6_OUTPUT_ROOT": os.environ["PFI_P10_6_OUTPUT_ROOT"],
})


{'PFI_RSNA_ROOT': '/content/RSNA_LUMBAR_DISC', 'PFI_RSNA_TRAIN_IMAGES': '/content/RSNA_LUMBAR_DISC/train_images', 'PFI_P10_6_OUTPUT_ROOT': '/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings'}


In [28]:
# 8) Repositorio y módulo AI
PFI_REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
PFI_REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))
PFI_REPO_REF = os.getenv("PFI_REPO_REF", "enzo/p10-6-ai-rsna-findings")
if not (PFI_REPO_ROOT / "ai_service" / "pfi_ai_service").exists():
    subprocess.check_call(["git", "clone", "--branch", PFI_REPO_REF, "--single-branch", PFI_REPO_URL, str(PFI_REPO_ROOT)])
else:
    subprocess.check_call(["git", "fetch", "origin"], cwd=PFI_REPO_ROOT)
    subprocess.check_call(["git", "checkout", PFI_REPO_REF], cwd=PFI_REPO_ROOT)
    subprocess.check_call(["git", "pull", "--ff-only"], cwd=PFI_REPO_ROOT)
repo_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PFI_REPO_ROOT, text=True).strip()
sys.path.insert(0, str(PFI_REPO_ROOT / "ai_service"))
from pfi_ai_service.training.rsna_preflight import build_config, run_preflight, runtime_versions, set_reproducible_seed, validate_dataset_structure
print({"repoRef": PFI_REPO_REF, "repoSha": repo_sha})


{'repoRef': 'enzo/p10-6-ai-rsna-findings', 'repoSha': 'ef931bb0cf56e7c9c45b72658c44b79daa890141'}


In [29]:
# 9) Configuración reproducible
CFG = build_config()
set_reproducible_seed(CFG.seed)
print(json.dumps({
    "seed": CFG.seed, "syntheticMode": CFG.synthetic,
    "rsnaRootConfigured": str(CFG.rsna_root),
    "trainImagesConfigured": str(CFG.train_images),
    "outputRootConfigured": str(CFG.output_root),
    "modelRootConfigured": str(CFG.model_root),
    "humanReviewRequired": True, "notClinicalDiagnosis": True,
    "runtime": runtime_versions(),
}, indent=2))


{
  "seed": 2026,
  "syntheticMode": false,
  "rsnaRootConfigured": "/content/RSNA_LUMBAR_DISC",
  "trainImagesConfigured": "/content/RSNA_LUMBAR_DISC/train_images",
  "outputRootConfigured": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings",
  "modelRootConfigured": "/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings",
  "humanReviewRequired": true,
  "notClinicalDiagnosis": true,
  "runtime": {
    "python": "3.12.13",
    "pytorch": "2.11.0+cpu",
    "pydicom": "3.0.2",
    "simpleitk": "2.5.6",
    "monai": "1.6.0",
    "timm": "1.0.28",
    "cudaAvailable": false,
    "cudaVersion": null,
    "gpu": null
  }
}


In [30]:
# 10) Preflight de estructura train-only
structure = validate_dataset_structure(CFG) if not CFG.synthetic else {"syntheticMode": True, "officialTestAccessed": False}
if structure.get("officialTestAccessed") is not False:
    raise RuntimeError("El preflight no puede acceder al test oficial.")
print(json.dumps(structure, indent=2))


{
  "rsnaRoot": "<path:d52ccd5a05eb>",
  "trainImagesRoot": "<path:a3230285645f>",
  "requiredPresent": [
    "train.csv",
    "train_images",
    "train_label_coordinates.csv",
    "train_series_descriptions.csv"
  ],
  "officialTestPresent": false,
  "officialTestAccessed": false
}


In [31]:
# 11) Ejecutar inventario y reportes externos
started = time.time()
summary = run_preflight(CFG)
elapsed = time.time() - started
if summary["officialTestAccessed"] is not False:
    raise RuntimeError("officialTestAccessed debe permanecer false.")
if not summary["humanReviewRequired"] or not summary["notClinicalDiagnosis"]:
    raise RuntimeError("Gobernanza inválida para P10.6-AI.")
print(json.dumps({
    "durationMinutes": round(elapsed/60,2), "nStudies": summary["nStudies"],
    "nSeries": summary["nSeries"], "nDicom": summary["nDicom"],
    "officialTestPresent": summary["officialTestPresent"],
    "officialTestAccessed": summary["officialTestAccessed"],
    "outputs": summary["outputs"],
}, indent=2))


{
  "durationMinutes": 2.63,
  "nStudies": 1975,
  "nSeries": 6294,
  "nDicom": 147218,
  "officialTestPresent": false,
  "officialTestAccessed": false,
  "outputs": {
    "dataset_inventory.json": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/dataset_inventory.json",
    "rsna_preflight_report.json": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/rsna_preflight_report.json",
    "rsna_preflight_report.md": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/rsna_preflight_report.md",
    "series_inventory.csv": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/series_inventory.csv",
    "label_distribution.csv": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/label_distribution.csv",
    "sequence_availability.csv": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/sequence_availability.csv",
    "coordinate_validation.csv": "/content/drive/MyDrive/PFI_MVP/results/P10_6_rsna_findings/coordinate_validation.csv",
    "mi

In [32]:
# 12) Evidencia sanitizada para Notebook 54
print(json.dumps({
    "dataset": summary["dataset"], "csvSha256": summary["csvSha256"],
    "sequenceAvailability": summary["sequenceAvailability"],
    "coordinateIssues": summary["coordinateIssues"],
    "futureMetrics": summary["futureMetrics"], "limitations": summary["limitations"],
    "humanReviewRequired": summary["humanReviewRequired"],
    "notClinicalDiagnosis": summary["notClinicalDiagnosis"],
    "officialTestAccessed": summary["officialTestAccessed"],
    "nextNotebook": "54_internal_split_and_model_plan",
}, indent=2))


{
  "dataset": "RSNA_LumbarDISC",
  "csvSha256": {
    "train.csv": "f0c9e06486bcddcd83b1ee1d95ab9db8e028d7c3c9fcbada950f0b8e4d828528",
    "train_label_coordinates.csv": "416fb434b5bdbf69814210d03dd791dff5ef9a010bac004f5f68e8b79e852635",
    "train_series_descriptions.csv": "bf8cc1aa55e4b5536b2f0fa8a060fc1912faeb65f9ba422d9d91b92a947b1c33"
  },
  "sequenceAvailability": {
    "studiesWithSagittalT1": 1973,
    "studiesWithSagittalT2Stir": 1974,
    "studiesWithAxialT2": 1975
  },
  "coordinateIssues": {
    "rows": 48692,
    "missingDicom": 0,
    "outsideImage": 0,
    "sensitiveHeaderRows": 48692
  },
  "futureMetrics": [
    "recall_sensitivity",
    "precision",
    "specificity",
    "macro_f1",
    "balanced_accuracy",
    "roc_auc_one_vs_rest",
    "confusion_matrix",
    "weighted_log_loss",
    "calibration",
    "severe_false_negatives",
    "metrics_by_level"
  ],
  "limitations": [
    "No final training was started.",
    "Official test, if present, was not accessed.",
 